# Why delegate at all

**Scenario:** a screening assistant works one staff backend requisition. For every candidate it reads
a raw applicant packet, then answers one question. Does this person clear the bar.

The packets are noise and the answer is one line. Three candidates in, the parent thread is carrying
every packet it has read, and paying for all of them on every turn.

Think of it as a research assistant with an index card. The assistant reads the file. You get the
card.

## Mechanics

A conversation is a list you resend in full every turn, so anything appended is a bill you keep
paying.

| Thing | Where it lives | What the next turn pays for it |
|---|---|---|
| `messages` | a list in your process | the whole list is sent again |
| `usage.prompt_tokens` | on every response | the exact size of the list you just sent |
| a sub-task run inline | appended to `messages` | its input and its output, on every later turn |
| a sub-task run as a subagent | a second list you throw away | only the line you copy back |

`usage.prompt_tokens` settles the argument, because the provider counts it and you do not. A subagent
is a second agent run with its own list, and this is the whole reason to want one.

## The picture

![The packet stays on the far side of the boundary and only a line crosses back](images/inline-versus-isolated.svg)

Same work either way. The difference is which side of the boundary the noise stays on.

## The cost

```
carried = prompt_tokens_per_turn x turns in the session
```

Reading a packet is paid once. Keeping it is paid on every turn after that.

## The failure

The packets are built here rather than fetched, so the lesson runs the same way every time.

In [1]:
CANDIDATES = [("C-1001", 7), ("C-1002", 3), ("C-1003", 9)]


def packet(candidate_id, years):
    """One raw applicant packet, the way a job board hands it over."""
    lines = [f"APPLICANT {candidate_id} | source: job-board-feed | status: new"]
    for week in range(1, 19):
        lines.append(f"  w{week:02d} page_view jd=staff-backend dwell=41s referrer=aggregator")
        lines.append(f"  w{week:02d} profile_edit field=summary chars=180 autosave=true")
    lines.append(f"  self_reported: {years} years python, "
                 f"{max(years - 1, 0)} years distributed systems")
    lines.append("  cover_letter: I am passionate about scale and enjoy mentoring juniors.")
    lines.append("  employment: three roles, no gaps over two months, references on file.")
    return "\n".join(lines)


print(f"one packet is {len(packet('C-1001', 7))} characters")

one packet is 2421 characters


Now the screening step, written the way most teams write it first. The packet goes straight into the
thread the assistant is already holding.

In [2]:
from vault import get_client, load_env, model_for

load_env()
client = get_client("05-subagent-delegation/01-why-delegate-at-all")

SYSTEM = ("You screen candidates for staff backend requisition REQ-77. "
          "The bar is five years of Python. Answer in one short line.")
ASK = "Read the raw packet below and say whether this candidate clears the bar.\n"


def screen_inline(thread, candidate_id, years):
    """Append the packet to the parent thread, then ask. The thread keeps growing."""
    thread.append({"role": "user", "content": ASK + packet(candidate_id, years)})
    reply = client.chat.completions.create(
        model=model_for("default"), max_tokens=120, messages=thread)
    thread.append({"role": "assistant", "content": reply.choices[0].message.content})
    return reply

Three candidates, then the question the session exists to answer. Watch the token count, and watch
the answers.

In [3]:
PARENT_BUDGET_TOKENS = 400

thread = [{"role": "system", "content": SYSTEM}]
for candidate_id, years in CANDIDATES:
    reply = screen_inline(thread, candidate_id, years)
    print(f"{candidate_id} ({years} yrs): parent thread {reply.usage.prompt_tokens:>5} tokens"
          f"   -> {(reply.choices[0].message.content or '').strip()[:42]}")

thread.append({"role": "user", "content": "Which of the three do we advance? One line."})
final = client.chat.completions.create(
    model=model_for("default"), max_tokens=120, messages=thread)
print(f"\nfinal question : {final.usage.prompt_tokens} tokens of context")
print(f"answer         : {(final.choices[0].message.content or '').strip()}")
assert final.usage.prompt_tokens <= PARENT_BUDGET_TOKENS, (
    f"parent context reached {final.usage.prompt_tokens} tokens "
    f"for {len(CANDIDATES)} candidates")

C-1001 (7 yrs): parent thread   868 tokens   -> This candidate does not clear the bar.
C-1002 (3 yrs): parent thread  1717 tokens   -> This candidate does not clear the bar.
C-1003 (9 yrs): parent thread  2566 tokens   -> This candidate clears the bar.

final question : 2583 tokens of context
answer         : We advance C-1003.


AssertionError: parent context reached 2583 tokens for 3 candidates

## The diagnosis

The assertion fires, and the number it prints is the lesson.

`usage.prompt_tokens` is the size of the list you sent. The packets are in that list, so the parent
pays for all three on every turn. The reading was done once. The carrying is done always.

Read the verdicts too. The candidate with seven years was called short of a five year bar, and the
final answer names one candidate when two qualify. The packets were never the deliverable.

## The fix

Give the reading to a subagent. It gets its own list, reads the packet, returns one line, and its
list is thrown away.

In [4]:
SUB_SYSTEM = ("The bar is five years of Python. Read the packet. "
              "Reply with exactly one line: years_python=<n>; clears_bar=<yes|no>.")


def screen_isolated(candidate_id, years):
    """A subagent with its own thread. Only the line it returns crosses back."""
    reply = client.chat.completions.create(
        model=model_for("default"), max_tokens=60,
        messages=[{"role": "system", "content": SUB_SYSTEM},
                  {"role": "user", "content": packet(candidate_id, years)}])
    return (reply.choices[0].message.content or "").strip(), reply.usage.prompt_tokens

Same candidates, same packets, same model. The only change is which list the packet lands in.

In [5]:
parent = [{"role": "system", "content": SYSTEM}]
read_tokens = 0
for candidate_id, years in CANDIDATES:
    note, tokens = screen_isolated(candidate_id, years)
    read_tokens += tokens
    parent.append({"role": "user", "content": f"screener report {candidate_id}: {note}"})
    print(f"{candidate_id}: subagent read {tokens} tokens, handed back {len(note)} characters")

parent.append({"role": "user", "content": "Which of the three do we advance? One line."})
after = client.chat.completions.create(
    model=model_for("default"), max_tokens=120, messages=parent)
print(f"\nsubagents read : {read_tokens} tokens, none of it in the parent")
print(f"before         : {final.usage.prompt_tokens} tokens of parent context")
print(f"after          : {after.usage.prompt_tokens} tokens of parent context")
print(f"answer         : {(after.choices[0].message.content or '').strip()}")

C-1001: subagent read 858 tokens, handed back 30 characters
C-1002: subagent read 858 tokens, handed back 30 characters
C-1003: subagent read 858 tokens, handed back 30 characters

subagents read : 2574 tokens, none of it in the parent
before         : 2583 tokens of parent context
after          : 104 tokens of parent context
answer         : C-1001 and C-1003.


The work did not shrink. The subagents read every character. What shrank is what the parent carries
into the next turn, and a long session multiplies that. Prices change without notice, so read them
from the probe rather than typing one in.

In [6]:
from vault import provider_truth

rate = float(provider_truth()["models"][model_for("default")]["prompt_usd_per_token"])
turns = 40

for label, tokens in (("inline", final.usage.prompt_tokens),
                      ("isolated", after.usage.prompt_tokens)):
    print(f"{label:9} {tokens:>5} tokens per parent turn"
          f"   {tokens * rate * turns:.5f} usd over a {turns} turn shift")

inline     2583 tokens per parent turn   0.01033 usd over a 40 turn shift
isolated    104 tokens per parent turn   0.00042 usd over a 40 turn shift


## The gate

The regression to prevent is somebody pasting a packet back into the parent because it was easier
than passing the report. This needs no model.

In [7]:
def test_no_packet_line_ever_reaches_the_parent():
    body = "".join(message["content"] for message in parent)
    assert "page_view" not in body, "a raw packet line is sitting in the parent thread"
    assert after.usage.prompt_tokens <= PARENT_BUDGET_TOKENS, (
        f"parent context is {after.usage.prompt_tokens} tokens, "
        f"budget is {PARENT_BUDGET_TOKENS}")


test_no_packet_line_ever_reaches_the_parent()
print("gate holds: no packet line in the parent, and the budget that failed now passes")

gate holds: no packet line in the parent, and the budget that failed now passes


Append a packet to `parent` instead of the report and this test fails on the first line.

### Enterprise exploration

- A shift screens two hundred candidates. When does the parent thread stop fitting, and what would
  you measure to see it coming?
- The report is one line. What happens when a subagent returns a paragraph, and who notices?
- Screening is regulated in several places. If the packet never reaches the parent, where is the
  audit trail showing what the decision rested on?
- The subagents ran one after another. What is the cost and latency trade off of running them at
  once?

### Key takeaways

- `usage.prompt_tokens` is the size of the list you sent, so anything appended is paid for again.
- A subagent does the same work in a list you throw away.
- The parent grows by the report, not by the work.
- Print the number before and after. The measurement is the argument.